# 00. Collect Current MAL User Ratings

This is an experimental pipeline for building a more current user-rating interaction layer from public MAL user lists.

It is intentionally separate from `02_get_user_ratings.ipynb`, which builds the stable Kaggle ratings file. This notebook writes to:

```text
data/processed/current_user_ratings.csv
```

The output schema is compatible with the rest of the project:

```text
userID, animeID, rating
```

## Privacy and access rule

This notebook does not keep a username database. Usernames are discovered from public MAL pages, processed immediately, converted to salted anonymous numeric `userID`s, and then discarded. Checkpoints store only hashes, counts, source progress, and errors.

Use public lists only. Private or access-controlled lists should be collected only with explicit consent and OAuth access.


In [ ]:
import hashlib
import json
import os
import re
import time
from datetime import datetime
from pathlib import Path
from urllib.parse import quote, unquote

import pandas as pd
import requests

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

BUILD_DIR = BASE_DIR / "data" / "build"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
SECRETS_DIR = BASE_DIR / "secrets"

for directory in [BUILD_DIR, PROCESSED_DIR, SECRETS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CURRENT_RATINGS_FILE = PROCESSED_DIR / "current_user_ratings.csv"
CURRENT_RATINGS_SUMMARY_FILE = BUILD_DIR / "current_user_ratings_summary.json"
CURRENT_RATINGS_CHECKPOINT_FILE = BUILD_DIR / "current_user_ratings_checkpoint.json"
CURRENT_RATINGS_FAILED_FILE = BUILD_DIR / "current_user_ratings_failed_users.json"
ANIME_DATASET_FILE = PROCESSED_DIR / "anime_dataset.csv"

MAL_API_BASE = "https://api.myanimelist.net/v2"
MAL_WEB_BASE = "https://myanimelist.net"

# Large-run defaults. Club discovery is the main reservoir; reviews and recommendations are fallback sources.
MAX_USERS_TO_PROCESS = 1_000_000
MAX_REVIEW_PAGES = 50
MAX_RECOMMENDATION_PAGES = 50
CLUB_INDEX_PAGES_BY_SORT = {
    "largest": {"sort": 5, "pages": 25},
    "recent_comment": {"sort": 2, "pages": 25},
}
MAX_CLUBS_TO_SCAN = None
MAX_CLUB_MEMBER_PAGES_PER_CLUB = None

DISCOVERY_SOURCE_ORDER = ["clubs", "recommendations", "reviews"]

# Discovery sources. Reviews and recommendations find active users beyond clubs.
DISCOVER_FROM_REVIEWS = True
DISCOVER_FROM_RECOMMENDATIONS = True
DISCOVER_FROM_CLUBS = True

FILTER_TO_CURRENT_CATALOG = True
RESET_OUTPUT = False

MAL_API_PAGE_LIMIT = 1000
MAL_API_DELAY_SECONDS = 1.0
PUBLIC_PAGE_DELAY_SECONDS = 2.5
MAX_RETRIES = 4
MIN_RATINGS_PER_USER = 11

REQUEST_HEADERS = {
    "User-Agent": "anime-recommender-course-project/1.0 (+local research notebook)",
}

print("Working directory:", BASE_DIR)
print("Experimental ratings output:", CURRENT_RATINGS_FILE)


## Local Secrets

Use ignored local files or environment variables. Do not put credentials directly in the notebook.

Supported local files:

```text
secrets/secret.txt
secrets/mal_client_id.txt
secrets/mal_access_token.txt
```

Recommended `secret.txt` format:

```text
MAL_CLIENT_ID=...
ANIDB_CLIENT=...
ANIDB_CLIENTVER=...
```

The parser also accepts `MAL_ClientID = ...`, so small spacing/casing differences should not break the run.

For public user lists, `MAL_CLIENT_ID` should usually be enough. If MAL rejects a list with client-id auth, use OAuth via `MAL_ACCESS_TOKEN` for accounts you are allowed to access.


In [ ]:
def now_iso():
    return datetime.now().isoformat(timespec="seconds")


def normalize_secret_key(value):
    return re.sub(r"[^A-Z0-9]+", "_", str(value or "").strip().upper()).strip("_")


def parse_secret_file(path):
    parsed = {}
    bare_values = []
    if not path.exists():
        return parsed, bare_values

    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" in line:
            key, value = line.split("=", 1)
            parsed[normalize_secret_key(key)] = value.strip().strip('"').strip("'")
        elif ":" in line:
            key, value = line.split(":", 1)
            parsed[normalize_secret_key(key)] = value.strip().strip('"').strip("'")
        else:
            bare_values.append(line)
    return parsed, bare_values


SECRET_CACHE = {}
SECRET_BARE_VALUES = []
for secret_path in [SECRETS_DIR / "secret.txt"]:
    parsed, bare_values = parse_secret_file(secret_path)
    SECRET_CACHE.update(parsed)
    SECRET_BARE_VALUES.extend(bare_values)


def read_secret(*names, default=None, filename=None, bare_index=None):
    for name in names:
        env_value = os.getenv(name)
        if env_value:
            return env_value.strip()

    if filename:
        path = SECRETS_DIR / filename
        if path.exists():
            return path.read_text(encoding="utf-8").strip()

    for name in names:
        value = SECRET_CACHE.get(normalize_secret_key(name))
        if value:
            return value

    if bare_index is not None and len(SECRET_BARE_VALUES) > bare_index:
        return SECRET_BARE_VALUES[bare_index]

    return default


MAL_ACCESS_TOKEN = read_secret("MAL_ACCESS_TOKEN", filename="mal_access_token.txt")
MAL_CLIENT_ID = read_secret("MAL_CLIENT_ID", "MAL_CLIENTID", "MAL_CLIENT", "CLIENT_ID", filename="mal_client_id.txt", bare_index=0)
CURRENT_USER_RATINGS_SALT = read_secret(
    "CURRENT_USER_RATINGS_SALT",
    filename="current_user_ratings_salt.txt",
    default="anime_recommender_current_ratings_v1",
)


def mal_headers():
    headers = dict(REQUEST_HEADERS)
    if MAL_ACCESS_TOKEN:
        headers["Authorization"] = f"Bearer {MAL_ACCESS_TOKEN}"
    elif MAL_CLIENT_ID:
        headers["X-MAL-CLIENT-ID"] = MAL_CLIENT_ID
    else:
        raise RuntimeError(
            "Missing MAL credentials. Set MAL_CLIENT_ID or MAL_ACCESS_TOKEN, "
            "or create secrets/secret.txt / secrets/mal_client_id.txt."
        )
    return headers


print("MAL auth mode:", "OAuth access token" if MAL_ACCESS_TOKEN else "Client ID" if MAL_CLIENT_ID else "missing")


## Checkpoint and Anonymization

The checkpoint does not store usernames. It stores salted username hashes so rediscovered users can be skipped without retaining the original username.


In [ ]:
def atomic_write_json(path, payload):
    path = Path(path)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    with tmp_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    tmp_path.replace(path)


def username_hash(username):
    text = f"{CURRENT_USER_RATINGS_SALT}:{username.strip().casefold()}"
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def anonymized_user_id(username):
    return int(username_hash(username)[:15], 16) % 2_147_483_647


def load_checkpoint():
    if not CURRENT_RATINGS_CHECKPOINT_FILE.exists():
        return {
            "completed_user_hashes": {},
            "failed_user_hashes": {},
            "source_state": {},
            "updated_at": None,
        }
    with CURRENT_RATINGS_CHECKPOINT_FILE.open("r", encoding="utf-8") as f:
        payload = json.load(f)
    payload.setdefault("completed_user_hashes", {})
    payload.setdefault("failed_user_hashes", {})
    payload.setdefault("source_state", {})
    return payload


def save_checkpoint(payload):
    payload["updated_at"] = now_iso()
    atomic_write_json(CURRENT_RATINGS_CHECKPOINT_FILE, payload)


def save_failed_user_registry(payload):
    failed_payload = {
        "updated_at": now_iso(),
        "note": "Usernames are not stored. Retry is possible only if the public user is rediscovered.",
        "failed_user_hashes": payload.get("failed_user_hashes", {}),
    }
    atomic_write_json(CURRENT_RATINGS_FAILED_FILE, failed_payload)


checkpoint = load_checkpoint()
completed_hashes = checkpoint["completed_user_hashes"]
failed_hashes = checkpoint["failed_user_hashes"]
source_state = checkpoint["source_state"]
save_failed_user_registry(checkpoint)
print({"completed_hashes": len(completed_hashes), "failed_hashes": len(failed_hashes)})


## Public Username Discovery

The official MAL API does not provide a public user-discovery endpoint. To build a current sample, this notebook discovers public usernames from MAL web pages, then uses the official MAL API for list data.

Discovery sources are ordered for volume first:

- club member pages from large clubs: `clubs.php?sort=5`
- club member pages from recently active clubs: `clubs.php?sort=2`
- recent anime recommendations
- recent anime reviews

No discovered username list is saved.

In [ ]:
PROFILE_RE = re.compile(r"/profile/([A-Za-z0-9_\-]+)")
CLUB_ID_RE = re.compile(r"(?:clubid=|cid=|/club/)(\d+)")


def request_text(url):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
        if response.status_code == 200:
            return response.text
        last_error = f"HTTP {response.status_code}: {response.text[:200]}"
        if response.status_code in {429, 500, 502, 503, 504}:
            sleep_seconds = min(120, (2 ** attempt) * PUBLIC_PAGE_DELAY_SECONDS)
            print(f"Retryable public page error {last_error}; sleeping {sleep_seconds:.1f}s")
            time.sleep(sleep_seconds)
            continue
        raise RuntimeError(last_error)
    raise RuntimeError(last_error or "public page request failed")


def extract_profile_usernames(html):
    names = []
    seen = set()
    for match in PROFILE_RE.finditer(html or ""):
        username = unquote(match.group(1)).strip()
        key = username.casefold()
        if username and key not in seen:
            seen.add(key)
            names.append(username)
    return names


# MAL club index links commonly use clubs.php?cid=20081.
# We intentionally do not match generic id= because club pages contain many unrelated ids.
def extract_club_ids(html):
    ids = []
    seen = set()
    for match in CLUB_ID_RE.finditer(html or ""):
        club_id = int(match.group(1))
        if club_id not in seen:
            seen.add(club_id)
            ids.append(club_id)
    return ids


def discover_from_review_pages():
    if not DISCOVER_FROM_REVIEWS:
        return
    completed_pages = set(source_state.get("review_pages_completed", []))
    for page in range(1, MAX_REVIEW_PAGES + 1):
        if page in completed_pages:
            continue
        url = f"{MAL_WEB_BASE}/reviews.php?t=anime&p={page}"
        print(f"discover reviews page {page}: {url}")
        html = request_text(url)
        for username in extract_profile_usernames(html):
            yield username
        completed_pages.add(page)
        source_state["review_pages_completed"] = sorted(completed_pages)
        save_checkpoint(checkpoint)
        time.sleep(PUBLIC_PAGE_DELAY_SECONDS)


def discover_from_recommendation_pages():
    if not DISCOVER_FROM_RECOMMENDATIONS:
        return
    completed_offsets = set(source_state.get("recommendation_offsets_completed", []))
    for offset in range(0, MAX_RECOMMENDATION_PAGES * 50, 50):
        if offset in completed_offsets:
            continue
        url = f"{MAL_WEB_BASE}/recommendations.php?s=recentrecs&t=anime&show={offset}"
        print(f"discover recommendations offset {offset}: {url}")
        html = request_text(url)
        for username in extract_profile_usernames(html):
            yield username
        completed_offsets.add(offset)
        source_state["recommendation_offsets_completed"] = sorted(completed_offsets)
        save_checkpoint(checkpoint)
        time.sleep(PUBLIC_PAGE_DELAY_SECONDS)


def discover_club_ids():
    discovered = list(source_state.get("club_ids_discovered", []))
    seen = set(discovered)

    completed_by_sort = source_state.setdefault("club_index_pages_completed_by_sort", {})
    legacy_completed = source_state.get("club_index_pages_completed", [])
    if legacy_completed and not completed_by_sort:
        completed_by_sort["legacy"] = legacy_completed

    for sort_name, config in CLUB_INDEX_PAGES_BY_SORT.items():
        sort_value = config["sort"]
        max_pages = config["pages"]
        completed_pages = set(completed_by_sort.get(sort_name, []))

        for page in range(1, max_pages + 1):
            if MAX_CLUBS_TO_SCAN is not None and len(discovered) >= MAX_CLUBS_TO_SCAN:
                break
            if page in completed_pages:
                continue

            url = f"{MAL_WEB_BASE}/clubs.php?sort={sort_value}&p={page}"
            print(f"discover club index sort={sort_name} page {page}: {url}")
            html = request_text(url)
            before = len(discovered)

            for club_id in extract_club_ids(html):
                if club_id not in seen:
                    seen.add(club_id)
                    discovered.append(club_id)
                if MAX_CLUBS_TO_SCAN is not None and len(discovered) >= MAX_CLUBS_TO_SCAN:
                    break

            completed_pages.add(page)
            completed_by_sort[sort_name] = sorted(completed_pages)
            source_state["club_index_pages_completed_by_sort"] = completed_by_sort
            source_state["club_ids_discovered"] = discovered
            save_checkpoint(checkpoint)
            print(f"  new club ids: {len(discovered) - before:,}; total discovered clubs: {len(discovered):,}")
            time.sleep(PUBLIC_PAGE_DELAY_SECONDS)

        if MAX_CLUBS_TO_SCAN is not None and len(discovered) >= MAX_CLUBS_TO_SCAN:
            break

    return discovered if MAX_CLUBS_TO_SCAN is None else discovered[:MAX_CLUBS_TO_SCAN]

def discover_from_club_member_pages():
    if not DISCOVER_FROM_CLUBS:
        return
    club_ids = discover_club_ids()
    completed_offsets = source_state.get("club_member_offsets_completed", {})
    completed_clubs = set(source_state.get("club_member_completed_club_ids", []))
    skipped_clubs = source_state.get("club_member_skipped_club_ids", {})

    for club_position, club_id in enumerate(club_ids, start=1):
        if club_id in completed_clubs or str(club_id) in skipped_clubs:
            continue
        done_offsets = set(completed_offsets.get(str(club_id), []))
        page_index = 0

        while True:
            if MAX_CLUB_MEMBER_PAGES_PER_CLUB is not None and page_index >= MAX_CLUB_MEMBER_PAGES_PER_CLUB:
                break

            offset = page_index * 36
            page_index += 1
            if offset in done_offsets:
                continue

            url = f"{MAL_WEB_BASE}/clubs.php?action=view&t=members&id={club_id}&show={offset}"
            print(f"discover club {club_position:,}/{len(club_ids):,} id={club_id} members offset {offset}: {url}")
            try:
                html = request_text(url)
            except RuntimeError as exc:
                reason = str(exc)[:300]
                if offset == 0:
                    skipped_clubs[str(club_id)] = {"reason": reason, "failed_at_offset": offset, "url": url}
                    source_state["club_member_skipped_club_ids"] = skipped_clubs
                    save_checkpoint(checkpoint)
                    print(f"  skipped club {club_id}: {reason}")
                else:
                    completed_clubs.add(club_id)
                    source_state["club_member_completed_club_ids"] = sorted(completed_clubs)
                    save_checkpoint(checkpoint)
                    print(f"  club {club_id} stopped at offset {offset}: {reason}")
                break

            names = extract_profile_usernames(html)

            yielded = 0
            for username in names:
                yielded += 1
                yield username

            done_offsets.add(offset)
            completed_offsets[str(club_id)] = sorted(done_offsets)
            source_state["club_member_offsets_completed"] = completed_offsets
            save_checkpoint(checkpoint)
            print(f"  candidate usernames on page: {yielded:,}")
            time.sleep(PUBLIC_PAGE_DELAY_SECONDS)

            if not names:
                completed_clubs.add(club_id)
                source_state["club_member_completed_club_ids"] = sorted(completed_clubs)
                save_checkpoint(checkpoint)
                print(f"  club {club_id} appears exhausted; marked complete")
                break


DISCOVERY_SOURCES = {
    "clubs": discover_from_club_member_pages,
    "recommendations": discover_from_recommendation_pages,
    "reviews": discover_from_review_pages,
}


def username_stream():
    seen_hashes = set()
    for source_name in DISCOVERY_SOURCE_ORDER:
        source = DISCOVERY_SOURCES[source_name]
        print(f"starting discovery source: {source_name}")
        for username in source():
            h = username_hash(username)
            if h in seen_hashes:
                continue
            seen_hashes.add(h)
            yield username


## Fetch Public Lists Through MAL API

For each discovered username, the notebook asks the official MAL API for completed anime list rows and keeps only ratings greater than zero.


In [ ]:
def request_json(url, params=None):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        response = requests.get(url, headers=mal_headers(), params=params, timeout=30)
        if response.status_code == 200:
            return response.json()

        if response.status_code in {401, 403}:
            raise RuntimeError(
                f"MAL auth/access error HTTP {response.status_code}. "
                "Try OAuth MAL_ACCESS_TOKEN if this public-list request is rejected."
            )
        if response.status_code == 404:
            raise RuntimeError("MAL user not found or animelist unavailable")

        last_error = f"HTTP {response.status_code}: {response.text[:300]}"
        if response.status_code in {429, 500, 502, 503, 504}:
            sleep_seconds = min(120, (2 ** attempt) * MAL_API_DELAY_SECONDS)
            print(f"Retryable MAL API error {last_error}; sleeping {sleep_seconds:.1f}s")
            time.sleep(sleep_seconds)
            continue
        raise RuntimeError(last_error)
    raise RuntimeError(last_error or "MAL API request failed")


def fetch_completed_ratings_for_user(username, catalog_ids=None):
    encoded_username = quote(username, safe="@")
    url = f"{MAL_API_BASE}/users/{encoded_username}/animelist"
    params = {
        "fields": "list_status",
        "status": "completed",
        "limit": MAL_API_PAGE_LIMIT,
        "offset": 0,
    }
    user_id = anonymized_user_id(username)
    rows = []
    page_count = 0

    while url:
        payload = request_json(url, params=params)
        page_count += 1

        for item in payload.get("data", []):
            node = item.get("node") or {}
            list_status = item.get("list_status") or {}
            anime_id = node.get("id")
            rating = list_status.get("score") or 0
            status = list_status.get("status")

            if status != "completed" or not rating or rating <= 0 or anime_id is None:
                continue
            if catalog_ids is not None and int(anime_id) not in catalog_ids:
                continue

            rows.append({"userID": user_id, "animeID": int(anime_id), "rating": int(rating)})

        url = (payload.get("paging") or {}).get("next")
        params = None
        if url:
            time.sleep(MAL_API_DELAY_SECONDS)

    return rows, page_count


## Run Collection

This cell is resumable. It checkpoints completed and failed users by hash only.

The current configuration is intended for a large run, up to `MAX_USERS_TO_PROCESS = 1_000_000`. At a one-second MAL API delay, a million attempted users is a multi-day job, so the checkpoint is part of the design rather than a convenience.

In [ ]:
if RESET_OUTPUT:
    if CURRENT_RATINGS_FILE.exists():
        CURRENT_RATINGS_FILE.unlink()
        print("Removed previous experimental ratings:", CURRENT_RATINGS_FILE)
    if CURRENT_RATINGS_CHECKPOINT_FILE.exists():
        CURRENT_RATINGS_CHECKPOINT_FILE.unlink()
        print("Removed previous checkpoint:", CURRENT_RATINGS_CHECKPOINT_FILE)
    checkpoint = load_checkpoint()
    completed_hashes = checkpoint["completed_user_hashes"]
    failed_hashes = checkpoint["failed_user_hashes"]
    source_state = checkpoint["source_state"]

catalog_ids = None
if FILTER_TO_CURRENT_CATALOG and ANIME_DATASET_FILE.exists():
    catalog_ids = set(pd.read_csv(ANIME_DATASET_FILE, usecols=["mal_id"])["mal_id"].dropna().astype(int))
    print(f"Catalog filter enabled: {len(catalog_ids):,} MAL ids")
else:
    print("Catalog filter disabled")

first_write = not CURRENT_RATINGS_FILE.exists() or CURRENT_RATINGS_FILE.stat().st_size == 0
processed_users_this_run = 0
new_rows_total = 0
started_at = time.time()

for username in username_stream():
    if MAX_USERS_TO_PROCESS is not None and processed_users_this_run >= MAX_USERS_TO_PROCESS:
        print("Reached MAX_USERS_TO_PROCESS")
        break

    h = username_hash(username)
    if h in completed_hashes:
        continue

    print(f"fetching discovered user hash={h[:10]}...")
    try:
        rows, page_count = fetch_completed_ratings_for_user(username, catalog_ids=catalog_ids)
    except Exception as exc:
        failed_hashes[h] = {
            "userID": anonymized_user_id(username),
            "error": str(exc),
            "last_attempt_at": now_iso(),
            "retryable": True,
        }
        save_checkpoint(checkpoint)
        save_failed_user_registry(checkpoint)
        print(f"  failed: {exc}")
        continue

    processed_users_this_run += 1

    if len(rows) >= MIN_RATINGS_PER_USER:
        out = pd.DataFrame(rows, columns=["userID", "animeID", "rating"])
        out.to_csv(
            CURRENT_RATINGS_FILE,
            mode="w" if first_write else "a",
            header=first_write,
            index=False,
        )
        first_write = False
        new_rows_total += len(out)
        written = len(out)
    else:
        written = 0

    completed_hashes[h] = {
        "completed_at": now_iso(),
        "ratings_seen": len(rows),
        "ratings_written": written,
        "pages_fetched": page_count,
        "userID": anonymized_user_id(username),
    }
    failed_hashes.pop(h, None)
    save_checkpoint(checkpoint)
    save_failed_user_registry(checkpoint)

    print(f"  ratings_seen={len(rows):,} written={written:,} pages={page_count}")
    time.sleep(MAL_API_DELAY_SECONDS)

elapsed_minutes = round((time.time() - started_at) / 60, 3)
source_state_summary = {
    "club_ids_discovered": len(source_state.get("club_ids_discovered", [])),
    "club_index_pages_completed_by_sort": {
        key: len(value) for key, value in source_state.get("club_index_pages_completed_by_sort", {}).items()
    },
    "club_member_clubs_started": len(source_state.get("club_member_offsets_completed", {})),
    "club_member_clubs_completed": len(source_state.get("club_member_completed_club_ids", [])),
    "recommendation_offsets_completed": len(source_state.get("recommendation_offsets_completed", [])),
    "review_pages_completed": len(source_state.get("review_pages_completed", [])),
}
summary = {
    "updated_at": now_iso(),
    "source": "MAL official API /users/{user_name}/animelist plus public MAL username discovery",
    "output_file": str(CURRENT_RATINGS_FILE),
    "processed_users_this_run": int(processed_users_this_run),
    "completed_user_hashes": len(completed_hashes),
    "failed_user_hashes": len(failed_hashes),
    "new_rows_written_this_run": int(new_rows_total),
    "catalog_filter_enabled": catalog_ids is not None,
    "catalog_ids": len(catalog_ids) if catalog_ids is not None else None,
    "elapsed_minutes": elapsed_minutes,
    "max_users_to_process": MAX_USERS_TO_PROCESS,
    "discovery_source_order": DISCOVERY_SOURCE_ORDER,
    "source_state_summary": source_state_summary,
    "columns": ["userID", "animeID", "rating"],
    "minimum_catalog_matched_ratings_per_user": MIN_RATINGS_PER_USER,
    "failed_user_registry": str(CURRENT_RATINGS_FAILED_FILE),
    "privacy_note": "Usernames are processed transiently. Checkpoints store salted hashes and anonymized userIDs, not usernames.",
}
atomic_write_json(CURRENT_RATINGS_SUMMARY_FILE, summary)
print("Summary saved:", CURRENT_RATINGS_SUMMARY_FILE)
print(summary)


## Inspect Output

The experimental file is separate from `ratings_processed.csv`. Review coverage before using it downstream.


In [ ]:
if CURRENT_RATINGS_FILE.exists():
    preview = pd.read_csv(CURRENT_RATINGS_FILE, nrows=10)
    print(preview)

    counts = pd.read_csv(CURRENT_RATINGS_FILE, usecols=["userID", "animeID", "rating"])
    print(
        {
            "rows": len(counts),
            "users": counts["userID"].nunique(),
            "anime": counts["animeID"].nunique(),
            "rating_min": int(counts["rating"].min()) if len(counts) else None,
            "rating_max": int(counts["rating"].max()) if len(counts) else None,
        }
    )
else:
    print("No current_user_ratings.csv created yet.")
